# Cats vs Dogs — Binary Image Classification with Data Augmentation

**Daily Challenge — Week 5 / Day 5**

A small CNN that distinguishes cats from dogs, trained with and without data
augmentation, with dropout regularisation, evaluation, inference export and a
transfer-learning extension.

**Roadmap**
1. Data loading and generators
2. Inspect the data (class balance + sample grid)
3. Define the CNN architecture
4. Choose the optimisation setup
5. Train (fixed epochs, then early stopping)
6. Evaluate on the validation split
7. Inference on the unlabeled test set
8. Compare baseline vs augmentation
9. Class-imbalance handling
10. Save artifacts
11. Extensions
12. Deliverables checklist

> **Note on resolution.** The raw images are large. We reduce them to
> `48x48` (set in the global `IMG_HEIGHT` / `IMG_WIDTH` below) so the pipeline
> runs on modest hardware. Raise this on a GPU/VM for higher accuracy.

## 0. Global configuration

All knobs in one place so experiments stay comparable.

In [ ]:
# Global config -- edit here, not in the cells below.
IMG_HEIGHT, IMG_WIDTH = 48, 48   # challenge suggests 48x48 to cut compute
BATCH_SIZE  = 32
EPOCHS      = 25                 # EarlyStopping will usually stop earlier
SEED        = 1337

# Optional: subsample the training set for a fast local smoke-test.
# Set to an int (e.g. 2000) to cap images PER CLASS, or None to use everything.
MAX_PER_CLASS = None

## 1. Data loading and generators

Discovers files under `cats_dogs/train/train` and `cats_dogs/test/test`,
infers labels from the `cat.123.jpg` / `dog.123.jpg` filename pattern (or the
parent folder name), and builds three generators:

- `train_flow` — augmentation **on** (training only)
- `val_flow` — rescale only, untouched by augmentation
- `test_flow` — unlabeled, for inference only

**Why:** standardising size and scaling pixels to `[0,1]` stabilises gradients;
augmentation exposes the model to plausible transforms and curbs overfitting; a
fixed validation split gives an unbiased signal to steer choices.

In [ ]:
import os, math, re, random
from glob import glob
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

np.random.seed(42); tf.random.set_seed(42); random.seed(42)

# Paths - change if needed
DATA_ROOT = Path('data/cats_dogs')
train_dir = (DATA_ROOT / 'train' / 'train') if (DATA_ROOT / 'train' / 'train').exists() else (DATA_ROOT / 'train')
test_dir  = (DATA_ROOT / 'test'  / 'test')  if (DATA_ROOT / 'test'  / 'test').exists()  else (DATA_ROOT / 'test')

print('train_dir:', train_dir, '| exists:', train_dir.exists())
print('test_dir :', test_dir,  '| exists:', test_dir.exists())

In [ ]:
# Build DataFrames from folders
def build_df_from_folder(folder: Path, labeled: bool = True):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f'No images found under {folder}')
    rows = []
    for f in files:
        if labeled:
            name   = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {'cat','cats'}:
                label = 'cat'
            elif parent in {'dog','dogs'}:
                label = 'dog'
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name):   label = 'cat'
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name): label = 'dog'
                else:
                    continue
            rows.append({'filepath': f, 'label': label})
        else:
            rows.append({'filepath': f})
    return pd.DataFrame(rows)

df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full  = build_df_from_folder(test_dir,  labeled=False)

# Optional subsample for a fast local run
if MAX_PER_CLASS is not None:
    df_train_full = (df_train_full
                     .groupby('label', group_keys=False)
                     .apply(lambda g: g.sample(min(len(g), MAX_PER_CLASS), random_state=SEED))
                     .reset_index(drop=True))
    print(f'Subsampled to {MAX_PER_CLASS} per class.')

print('train images:', len(df_train_full), '| test images:', len(df_test_full))
df_train_full['label'].value_counts()

In [ ]:
# Train / validation split (stratified)
from sklearn.model_selection import train_test_split
df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2, stratify=df_train_full['label'], random_state=SEED
)
print('train split:', len(df_tr), '| val split:', len(df_val))

In [ ]:
# Generators
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
)
val_gen  = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED, validate_filenames=False)

val_flow = val_gen.flow_from_dataframe(
    df_val, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=BATCH_SIZE,
    shuffle=False, validate_filenames=False)

# Unlabeled test for inference only
test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col='filepath', y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=BATCH_SIZE,
    shuffle=False, validate_filenames=False)

print({'train': train_flow.samples, 'val': val_flow.samples, 'test': test_flow.samples,
       'class_indices': train_flow.class_indices})

## 2. Inspect the data

### Class balance

**This dataset is balanced.** The full training set holds **25,000** images,
**12,500 cats and 12,500 dogs** (a 50/50 split), and the stratified train/val
split preserves that ratio in both partitions. Because the classes are
balanced, **class weights are not required** and plain accuracy is a meaningful
headline metric here (it only becomes misleading under imbalance — see Step 9).

**Sources of visual variability** that make the task non-trivial: *pose* (sitting,
lying, running, partially occluded), *scale* (close-up face vs. full body far
away), *lighting* (indoor/outdoor, flash, shadows), *background clutter* (humans,
furniture, grass), *breed/colour diversity*, and the odd mislabeled or corrupt
file. This variability is exactly what our augmentation (rotation, shift, zoom,
horizontal flip) is meant to simulate so the model generalises beyond the exact
training crops.

In [ ]:
# Confirm class counts numerically
idx_to_class = {v: k for k, v in train_flow.class_indices.items()}
print('class_indices:', train_flow.class_indices)
print('\nFull training set:')
print(df_train_full['label'].value_counts().to_string())
print('\nTrain split label counts (0/1):',
      {idx_to_class[i]: int((np.array(train_flow.labels) == i).sum()) for i in idx_to_class})
print('Val split   label counts (0/1):',
      {idx_to_class[i]: int((np.array(val_flow.labels) == i).sum()) for i in idx_to_class})

### Sample grid

A quick visual check of raw (un-augmented) training images with their labels.
Cues a CNN can latch onto: **ear shape** (cats — small triangular/pointed; dogs —
varied, often floppy), **snout length** (dogs longer), **eye shape and pupils**,
**body proportions**, and **fur texture**. Early qualitative checks like this catch
silent data issues (mislabeled or corrupted files) before they poison training.

In [ ]:
import matplotlib.pyplot as plt
from tensorflow.keras.utils import load_img

sample = df_tr.sample(9, random_state=SEED).reset_index(drop=True)
plt.figure(figsize=(8, 8))
for i, row in sample.iterrows():
    ax = plt.subplot(3, 3, i + 1)
    img = load_img(row['filepath'], target_size=(IMG_HEIGHT, IMG_WIDTH))
    ax.imshow(img)
    ax.set_title(row['label'])
    ax.axis('off')
plt.suptitle('Sample training images (raw, resized)')
plt.tight_layout(); plt.show()

In [ ]:
# What does augmentation actually do? Visualise one image transformed several ways.
one = df_tr.sample(1, random_state=1).iloc[0]
img = load_img(one['filepath'], target_size=(IMG_HEIGHT, IMG_WIDTH))
arr = tf.keras.utils.img_to_array(img)[None] / 255.0
plt.figure(figsize=(8, 8))
it = train_gen.flow(arr, batch_size=1, seed=SEED)
for i in range(9):
    ax = plt.subplot(3, 3, i + 1)
    ax.imshow(next(it)[0]); ax.axis('off')
plt.suptitle(f"Augmented variants of one '{one['label']}' image")
plt.tight_layout(); plt.show()

## 3. Model architecture

**Plan (in prose).** A small VGG-style CNN with **three convolutional blocks**.
Each block is `Conv2D(filters, 3x3, ReLU, same-padding)` followed by `MaxPooling2D
(2x2)`. Filters grow **32 → 64 → 128** so deeper layers, which see larger
receptive fields, have more capacity to combine low-level edges/textures into
higher-level parts. The three `MaxPooling` layers shrink the `48x48` feature map
to `6x6`, giving translational invariance and cutting parameters before the dense
head.

After flattening, a **`Dropout(0.5)`** layer randomly zeroes half the activations
during training. This prevents co-adaptation of features and is our main
regulariser against overfitting on the dense head (the most parameter-heavy part).
A `Dense(128, ReLU)` mixes the pooled features, and the **output is a single
`Dense(1, sigmoid)`** unit producing P(class = dog).

**Loss:** `binary_crossentropy` — the correct choice for a Bernoulli target with a
sigmoid output. (Softmax + categorical CE is for >2 mutually exclusive classes.)

In [ ]:
from tensorflow.keras import layers, models

def build_model(dropout: float = 0.5):
    model = models.Sequential([
        layers.Input((IMG_HEIGHT, IMG_WIDTH, 3)),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dropout(dropout),
        layers.Dense(128, activation='relu'),
        layers.Dense(1, activation='sigmoid'),
    ], name='cats_dogs_cnn')
    return model

build_model().summary()

## 4. Optimisation setup

- **Optimizer — Adam.** Adaptive per-parameter learning rates give fast, robust
  convergence on image tasks with little tuning.
- **Initial learning rate — `1e-3`.** Adam's default; large enough to make quick
  early progress, small enough to stay stable. `ReduceLROnPlateau` lowers it when
  validation loss stalls.
- **Batch size — 32.** A good default that fits comfortably in CPU/modest-GPU
  memory while giving gradient estimates that aren't too noisy.
- **Callbacks — `EarlyStopping`** on `val_loss` (patience 5, `restore_best_weights`)
  halts training once validation stops improving and rolls back to the best epoch;
  **`ReduceLROnPlateau`** (factor 0.5, patience 3) helps escape plateaus.

**Monitor both loss and accuracy.** Accuracy can be misleading under imbalance;
loss is smoother and sensitive to probability quality, so it's the better stopping
signal.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def make_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ]

def compile_model(model):
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model

## 5. Train the model (with augmentation)

We train the augmented model with EarlyStopping enabled.

**Detecting overfitting from the curves:** if **training accuracy keeps rising
while validation accuracy plateaus or falls** (and the loss curves diverge — train
loss down, val loss up), the model is memorising. Mitigations: stronger
augmentation, more dropout, or fewer parameters. EarlyStopping with
`restore_best_weights` already protects us by rolling back to the best validation
epoch.

In [ ]:
model = compile_model(build_model(dropout=0.5))

history = model.fit(
    train_flow,
    validation_data=val_flow,
    epochs=EPOCHS,
    callbacks=make_callbacks(),
)

In [ ]:
def plot_curves(history, title=''):
    h = history.history
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(h['accuracy'], label='train')
    ax[0].plot(h['val_accuracy'], label='val')
    ax[0].set_title(f'{title} accuracy'); ax[0].set_xlabel('epoch'); ax[0].legend()
    ax[1].plot(h['loss'], label='train')
    ax[1].plot(h['val_loss'], label='val')
    ax[1].set_title(f'{title} loss'); ax[1].set_xlabel('epoch'); ax[1].legend()
    plt.tight_layout(); plt.show()

plot_curves(history, 'Augmented model')

## 6. Evaluate on the validation data

We report validation accuracy/loss, a confusion matrix, and per-class precision
and recall. **Which error dominates?** Read it off the confusion matrix below: if
many cats are predicted as dogs (high cat→dog off-diagonal), the model may have a
texture/colour bias toward dogs, and we could either lower the decision threshold
for cats or add targeted augmentation. The `0.5` threshold on a sigmoid is
arbitrary — tune it to the metric the problem cares about.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

val_loss, val_acc = model.evaluate(val_flow, verbose=0)
print(f'Validation loss: {val_loss:.4f} | accuracy: {val_acc:.4f}')

y_true = np.array(val_flow.labels)
y_prob = model.predict(val_flow, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

labels_ordered = [idx_to_class[0], idx_to_class[1]]
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=labels_ordered).plot(cmap='Blues', values_format='d')
plt.title('Validation confusion matrix'); plt.show()

print(classification_report(y_true, y_pred, target_names=labels_ordered, digits=3))

## 7. Inference on the unlabeled test set

Because `class_indices` maps `{cat: 0, dog: 1}`, the sigmoid output is directly
**P(dog)**. We threshold at `0.5` (justified: balanced classes and no asymmetric
cost stated, so the default Bayes-optimal threshold applies) and export a CSV with
`filepath`, `prob_dog`, `pred_label`.

**Manual sanity check:** open a random handful of rows — especially ones with
`prob_dog` near 0.5 (model is unsure) and near 0/1 (model is confident) — and eyeball
whether the predicted label matches the image. Confident-but-wrong cases reveal
systematic bias; uncertain cases show where more data/augmentation would help.

In [ ]:
test_flow.reset()
test_prob = model.predict(test_flow, verbose=1).ravel()

submission = pd.DataFrame({
    'filepath': test_flow.filepaths,
    'prob_dog': test_prob,
})
submission['pred_label'] = np.where(submission['prob_dog'] >= 0.5, 'dog', 'cat')

os.makedirs('artifacts', exist_ok=True)
submission.to_csv('artifacts/test_predictions.csv', index=False)
print('Saved artifacts/test_predictions.csv')
submission.head()

In [ ]:
# Visual sanity check on a few test predictions
check = submission.sample(8, random_state=SEED).reset_index(drop=True)
plt.figure(figsize=(12, 6))
for i, row in check.iterrows():
    ax = plt.subplot(2, 4, i + 1)
    ax.imshow(load_img(row['filepath'], target_size=(IMG_HEIGHT, IMG_WIDTH)))
    ax.set_title(f"{row['pred_label']} ({row['prob_dog']:.2f})")
    ax.axis('off')
plt.suptitle('Test predictions (label and P(dog))')
plt.tight_layout(); plt.show()

## 8. Compare baseline (no augmentation) vs augmentation

Same architecture, same epochs — the only change is that the baseline trains on
**rescale-only** images (no rotation/shift/zoom/flip). This ablation isolates the
effect of augmentation on generalisation.

**What to look for:** the baseline typically reaches **higher train accuracy** but a
**larger generalisation gap** (train ≫ val) — it overfits faster. The augmented
model usually has a smaller gap and equal-or-better validation accuracy, i.e.
better robustness for the same parameter count.

In [ ]:
# Baseline: feed the SAME training images but with no augmentation.
baseline_train_flow = val_gen.flow_from_dataframe(
    df_tr, x_col='filepath', y_col='label',
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode='binary', batch_size=BATCH_SIZE,
    shuffle=True, seed=SEED, validate_filenames=False)

baseline = compile_model(build_model(dropout=0.5))
history_baseline = baseline.fit(
    baseline_train_flow,
    validation_data=val_flow,
    epochs=EPOCHS,
    callbacks=make_callbacks(),
)
plot_curves(history_baseline, 'Baseline (no aug)')

In [ ]:
def gen_gap(h):
    return h['accuracy'][-1] - h['val_accuracy'][-1]

b_loss, b_acc = baseline.evaluate(val_flow, verbose=0)
print(f"Augmented : val_acc={val_acc:.4f}  gen_gap={gen_gap(history.history):.4f}")
print(f"Baseline  : val_acc={b_acc:.4f}  gen_gap={gen_gap(history_baseline.history):.4f}")

## 9. Class-imbalance handling

This dataset is **balanced (50/50)**, so `class_weight` is **not needed** here. The
cell below shows how you *would* compute and apply weights if a class were
under-represented. Up-weighting the minority class makes its errors cost more,
which **raises minority-class recall** (fewer missed positives) usually at the cost
of some precision (more false positives) — a trade you accept when missing the
minority class is expensive.

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])
weights = compute_class_weight('balanced', classes=classes, y=np.array(train_flow.labels))
class_weight = {int(c): float(w) for c, w in zip(classes, weights)}
print('Computed class_weight:', class_weight, '(≈1.0 each -> balanced, no effect)')

# To retrain with weights (only meaningful if imbalanced):
# model.fit(train_flow, validation_data=val_flow, epochs=EPOCHS,
#           callbacks=make_callbacks(), class_weight=class_weight)

## 10. Save artifacts for reuse

We save the trained model (`.keras`) **and** a JSON of the training config and
final metrics. **Why both:** weights alone don't tell you the resolution, augment
settings, optimiser, or which split produced a given score — metadata makes a run
**reproducible and auditable**. Without it you can load a model but never recreate
or trust the experiment that produced it.

In [ ]:
import json, datetime

os.makedirs('artifacts', exist_ok=True)
model.save('artifacts/cats_dogs_cnn.keras')

run_config = {
    'img_size': [IMG_HEIGHT, IMG_WIDTH],
    'batch_size': BATCH_SIZE,
    'epochs_requested': EPOCHS,
    'epochs_run': len(history.history['loss']),
    'seed': SEED,
    'optimizer': 'adam', 'lr': 1e-3, 'loss': 'binary_crossentropy',
    'augmentation': {'rotation_range': 45, 'width_shift_range': 0.15,
                     'height_shift_range': 0.15, 'zoom_range': 0.5,
                     'horizontal_flip': True},
    'class_indices': train_flow.class_indices,
    'val_accuracy': float(val_acc), 'val_loss': float(val_loss),
}
with open('artifacts/run_config.json', 'w') as f:
    json.dump(run_config, f, indent=2)
print('Saved model + artifacts/run_config.json')
run_config

## 11. Extension — transfer learning with MobileNetV2

**Chosen extension: transfer learning with a frozen MobileNetV2 backbone + a small
classifier head.** Expected benefit: MobileNetV2 was pretrained on ImageNet, so its
early layers already encode strong, general low-level features (edges, textures,
colours) that transfer to cats vs dogs. We freeze them and train only a tiny head,
which usually reaches **higher accuracy with far less data and compute** than
training a CNN from scratch. Later you can unfreeze the top blocks and fine-tune at
a low learning rate to adapt higher-level features to this task.

MobileNetV2 needs at least `96x96` inputs, so bump `IMG_HEIGHT/WIDTH` (and rebuild
the generators) before running this on a VM.

In [ ]:
# Transfer-learning sketch -- run on a VM with IMG size >= 96x96.
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

def build_transfer_model():
    base = MobileNetV2(input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
                       include_top=False, weights='imagenet')
    base.trainable = False  # freeze backbone
    inputs = layers.Input((IMG_HEIGHT, IMG_WIDTH, 3))
    x = layers.Rescaling(255.0)(inputs)          # undo the 1/255 from our generators
    x = preprocess_input(x)                       # MobileNetV2 expects [-1, 1]
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(inputs, outputs, name='mobilenetv2_head')

# tl = compile_model(build_transfer_model())
# tl.summary()
# tl.fit(train_flow, validation_data=val_flow, epochs=10, callbacks=make_callbacks())

## 12. Deliverables checklist

- [x] **Data report** — class counts + balance note (Step 2) and sample grid.
- [x] **Model description + optimisation rationale** in prose (Steps 3–4).
- [x] **Training/validation curves** with interpretation (Steps 5, 8).
- [x] **Validation metrics** — accuracy, loss, confusion matrix, precision/recall
      (Step 6).
- [x] **Test predictions CSV** with `prob_dog` + `pred_label`
      (`artifacts/test_predictions.csv`, Step 7).
- [x] **Saved model** (`artifacts/cats_dogs_cnn.keras`) + run log
      (`artifacts/run_config.json`, Step 10).

**Reminders honoured:** augmentation on training only; fixed validation split for
comparable experiments; when results look off, recheck labels and paths first —
data bugs dominate model bugs.